In [5]:
%load_ext dotenv
%dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [6]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser

In [9]:
vectorstore = Chroma(persist_directory = "./intro-to-ds-lectures",
                                    embedding_function = OpenAIEmbeddings(model = "text-embedding-ada-002"))

In [10]:
len(vectorstore.get()['documents'])

23

In [11]:
retriever = vectorstore.as_retriever(search_type = 'mmr',
                                     search_kwargs = {'k': 3, 
                                                      'lambda_mult': 0.7})

In [12]:
TEMPLATE = '''
Answer the following question:
{question}

To answer the question, use only the following context:
{context}

At the end to the response, specify the name of the lecture this context is taken from in the format:
Resources: *Lecture Title*
where *Lecture Title* should be substituted with the title of all resource lectures.
'''

prompt_template = PromptTemplate.from_template(TEMPLATE)

In [13]:
chat = ChatOpenAI(model_name = 'gpt-4', 
                  seed = 365, 
                  max_tokens = 250)

In [17]:
question = "What software do data scientists use?"

In [24]:
chain = ({'context': retriever,
         'question': RunnablePassthrough()} 
         | prompt_template
         | chat 
         | StrOutputParser())

In [25]:
chain.invoke(question)

'Data scientists use a variety of software tools. The two most popular tools are R and Python, which are capable of manipulating data and are integrated with multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations but also adaptable to solve a range of business and data-related problems. Another notable software framework is Hadoop, which is designed to handle the complexity and computational intensity of big data by distributing computational tasks on multiple computers. Furthermore, software like Power BI, SaS, Qlik, and especially Tableau are used for business intelligence visualizations.\n\nResources: Programming Languages & Software Employed in Data Science - All the Tools You Need'